# Task 1: Evaluate base vs fine-tuned on the held-out test set (Colab T4)

**Needs on Drive** (`MyDrive/tndrr/task1_lora/`): `config.json`, `requirements-colab.txt`, `data/test.jsonl`,
and the `adapter/` folder written by `train_qlora.ipynb`.

Both models see the identical prompt and the identical 4-bit base weights: the base model is the
fine-tuned model with its LoRA adapter switched off (`disable_adapter()`), so the only difference is the adapter.

Decoding is pure greedy (`do_sample=False`, `repetition_penalty=1.0`; Qwen's default generation config
otherwise applies a repetition penalty), max 128 new tokens.

Metrics:
- **valid JSON %** - the raw output parses as a JSON object with `json.loads` (no cleanup).
- **exact match %** - the parsed object equals the expected object (all 6 fields, nothing extra).
- **per-field accuracy %** - for each field, parsed value == expected value. Invalid JSON or a missing key counts as wrong.
- **valid JSON after extracting {...} %** - lenient: the text between the first `{` and last `}` parses. Reported separately only to show how much of the base model's failure is formatting (e.g. markdown fences).

Outputs (Drive `results/`): `predictions.jsonl` (every raw output), `metrics.json`.

In [ ]:
# 1. Mount Drive, check inputs.
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/tndrr/task1_lora'
for f in ['config.json', 'requirements-colab.txt', 'data/test.jsonl', 'adapter/adapter_config.json']:
    assert os.path.exists(f'{BASE}/{f}'), f'missing {BASE}/{f}'
os.makedirs(f'{BASE}/results', exist_ok=True)
print('inputs found')

In [ ]:
# 2. Install pinned libraries (same as training).
!pip install -q -r /content/drive/MyDrive/tndrr/task1_lora/requirements-colab.txt

In [ ]:
# 3. Imports, config, seed.
import json, time, torch, transformers, peft, bitsandbytes
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import PeftModel

cfg = json.load(open(f'{BASE}/config.json'))
set_seed(cfg['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'
VERSIONS = {'torch': torch.__version__, 'transformers': transformers.__version__,
            'peft': peft.__version__, 'bitsandbytes': bitsandbytes.__version__}
print(GPU, VERSIONS)

In [ ]:
# 4. Load the 4-bit base model exactly as in training, then attach the trained adapter.
q = cfg['quantization']
bnb_config = BitsAndBytesConfig(
    load_in_4bit=q['load_in_4bit'],
    bnb_4bit_quant_type=q['bnb_4bit_quant_type'],
    bnb_4bit_use_double_quant=q['bnb_4bit_use_double_quant'],
    bnb_4bit_compute_dtype=getattr(torch, q['bnb_4bit_compute_dtype']),
)
tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'])
base = AutoModelForCausalLM.from_pretrained(
    cfg['base_model'], quantization_config=bnb_config, torch_dtype=torch.float16, device_map={'': 0})
model = PeftModel.from_pretrained(base, f'{BASE}/adapter')
model.eval()

In [ ]:
# 5. Prompt, generation and scoring helpers.
FIELDS = ['type', 'amount', 'currency', 'category', 'date', 'title']
MISSING = object()   # a key the model left out is never equal to anything, not even null

def prompt_text(row):
    # Identical to train_qlora.ipynb: chat template up to '<|im_start|>assistant\n'.
    msgs = [{'role': 'system', 'content': cfg['system_prompt']}, {'role': 'user', 'content': row['input']}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(row):
    enc = tokenizer(prompt_text(row), return_tensors='pt').to(model.device)
    out = model.generate(**enc, max_new_tokens=cfg['eval']['max_new_tokens'], do_sample=False,
                         repetition_penalty=1.0, temperature=None, top_p=None, top_k=None,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def parse_strict(text):
    try:
        obj = json.loads(text)
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

def parse_lenient(text):
    i, j = text.find('{'), text.rfind('}')
    return parse_strict(text[i:j + 1]) if i != -1 and j > i else None

def score(parsed, gold):
    fields = {f: parsed is not None and parsed.get(f, MISSING) == gold[f] for f in FIELDS}
    exact = parsed is not None and parsed == gold
    return fields, exact

In [ ]:
# 6. Run both models over every test example (one at a time, so no padding effects).
test = [json.loads(l) for l in open(f"{BASE}/{cfg['data']['test']}", encoding='utf-8')]
preds, start = [], time.time()
for i, row in enumerate(test):
    with model.disable_adapter():           # adapter OFF = the original base model
        base_raw = generate(row)
    ft_raw = generate(row)                  # adapter ON = fine-tuned
    rec = {'id': i, 'input': row['input'], 'expected': row['output'],
           'template_id': row['template_id'], 'unseen_word': row['unseen_word']}
    for name, raw in [('base', base_raw), ('finetuned', ft_raw)]:
        parsed = parse_strict(raw)
        fields, exact = score(parsed, row['output'])
        rec[name] = {'raw': raw, 'valid_json': parsed is not None,
                     'valid_json_lenient': parse_lenient(raw) is not None,
                     'exact_match': exact, 'fields': fields}
    preds.append(rec)
    if i % 10 == 0:
        print(i, '| base:', base_raw[:80].replace('\n', ' '), '| ft:', ft_raw[:80])
eval_sec = time.time() - start

with open(f'{BASE}/results/predictions.jsonl', 'w', encoding='utf-8') as f:
    for r in preds:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'done: {len(preds)} examples in {eval_sec:.0f}s')

In [ ]:
# 7. Aggregate metrics (all percentages over the full test set) and save.
def pct(xs):
    xs = list(xs)
    return round(100 * sum(xs) / len(xs), 1) if xs else None

def summarise(name, rows):
    m = [r[name] for r in rows]
    return {
        'n': len(rows),
        'valid_json_pct': pct(x['valid_json'] for x in m),
        'valid_json_lenient_pct': pct(x['valid_json_lenient'] for x in m),
        'exact_match_pct': pct(x['exact_match'] for x in m),
        'field_accuracy_pct': {f: pct(x['fields'][f] for x in m) for f in FIELDS},
    }

subsets = {
    'all': preds,
    'expense': [r for r in preds if r['expected']['type'] == 'expense'],
    'reminder': [r for r in preds if r['expected']['type'] == 'reminder'],
    'seen_words': [r for r in preds if not r['unseen_word']],
    'unseen_words': [r for r in preds if r['unseen_word']],
}
metrics = {
    'test_file': cfg['data']['test'],
    'decoding': {'do_sample': False, 'repetition_penalty': 1.0, 'max_new_tokens': cfg['eval']['max_new_tokens']},
    'gpu': GPU, 'versions': VERSIONS, 'eval_runtime_sec': round(eval_sec, 1),
    'base': {k: summarise('base', v) for k, v in subsets.items()},
    'finetuned': {k: summarise('finetuned', v) for k, v in subsets.items()},
}
json.dump(metrics, open(f'{BASE}/results/metrics.json', 'w'), indent=2)

print(f"{'metric':<28}{'base':>10}{'fine-tuned':>12}")
b, ft = metrics['base']['all'], metrics['finetuned']['all']
for k in ['valid_json_pct', 'valid_json_lenient_pct', 'exact_match_pct']:
    print(f'{k:<28}{str(b[k]):>10}{str(ft[k]):>12}')
for f in FIELDS:
    print(f'{"field: " + f:<28}{str(b["field_accuracy_pct"][f]):>10}{str(ft["field_accuracy_pct"][f]):>12}')
for k in ['expense', 'reminder', 'seen_words', 'unseen_words']:
    print(f'{"exact, " + k + " (n=" + str(metrics["base"][k]["n"]) + ")":<28}'
          f'{str(metrics["base"][k]["exact_match_pct"]):>10}{str(metrics["finetuned"][k]["exact_match_pct"]):>12}')

In [ ]:
# 8. Fine-tuned model's mistakes (real outputs), to choose 3 for results/failures.md.
wrong = [r for r in preds if not r['finetuned']['exact_match']]
print(len(wrong), 'fine-tuned mistakes')
for r in wrong:
    bad = [f for f, ok in r['finetuned']['fields'].items() if not ok]
    print(f"\n#{r['id']} {r['template_id']} unseen_word={r['unseen_word']} wrong fields={bad}")
    print('  input   :', r['input'])
    print('  expected:', json.dumps(r['expected'], ensure_ascii=False))
    print('  got     :', r['finetuned']['raw'])